# Desenvolvimento de modelos - IntentClassifier

### Exemplo de carregamento de embeddings como camada Keras

In [ ]:
! pip3 install tensorflow_text==2.19.0 wandb --quiet

In [ ]:
import tensorflow_hub as hub
import tensorflow_text

embeddings_layer = hub.KerasLayer('https://tfhub.dev/google/universal-sentence-encoder-multilingual/3')
embeddings = embeddings_layer([
    "The rain in Brazil.",
    "A chuva no Brazil",
    "mainly",
    "principalmente"
])

print(embeddings.shape)  #(4,128)

(4, 512)


In [ ]:
# Calcular a similaridade do embeddings[0] e embeddings[1]
similarity = tf.reduce_sum(tf.multiply(embeddings[0], embeddings[1]))
print(similarity)

## Exemplo: IntentClassifier

In [ ]:
import os
import logging
from pathlib import Path
from typing import List, Optional, Union, Tuple, Dict, Any
from dataclasses import dataclass
import yaml
from pprint import pprint
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, cohen_kappa_score

import tensorflow as tf
from tensorflow.keras import regularizers
import tensorflow_text
import tensorflow_hub as hub
from tensorflow.keras.saving import register_keras_serializable

import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbEvalCallback # WandbModelCheckpoint

import dotenv
dotenv.load_dotenv()

logger = logging.getLogger(__name__)

@register_keras_serializable()
class HubLayer(tf.keras.layers.Layer):
    """
    A custom Keras layer to load and use a TensorFlow Hub module.

    This layer loads a pre-trained model from a TensorFlow Hub URL
    and integrates it into a Keras model. It can be set to be
    trainable or frozen.

    :param hub_url: The URL of the TensorFlow Hub module to load.
    :type hub_url: str
    :param trainable: Whether the loaded Hub module should be trainable.
    :type trainable: bool, optional
    """
    def __init__(self, hub_url, trainable=False, **kwargs):
        """
        Initializes the HubLayer.
        """
        super(HubLayer, self).__init__(**kwargs)
        self.hub_module = hub.load(hub_url)
        self.hub_module.trainable = trainable

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """
        Executes the forward pass of the layer.

        :param inputs: The input tensor(s) to the Hub module.
        :type inputs: tf.Tensor
        :return: The output tensor(s) from the Hub module.
        :rtype: tf.Tensor
        """
        return self.hub_module(inputs)

@dataclass
class Config:
    """
    A dataclass to hold all configuration parameters for the IntentClassifier.

    This object stores settings related to the dataset, model architecture,
    training process, and logging.
    """
    dataset_name: str = "undefined"
    """Name of the dataset, used for logging and model naming."""
    codes : List[str] = None
    """A list of intent codes (class labels). Automatically populated from data if not provided."""
    architecture: str = "v0.1.5"
    """Version tag for the model architecture."""
    task: str = "undefined"
    """The current task being performed (e.g., 'train', 'predict')."""
    stop_words_file: Optional[str] = None
    """Path to a text file containing stopwords, one per line."""
    min_words: int = 1
    """The minimum number of words required in an utterance for processing. Shorter inputs are padded."""
    embedding_model: Union[str, List[str]] = 'https://www.kaggle.com/models/google/universal-sentence-encoder/tensorFlow2/multilingual/2'
    """URL or path to the TensorFlow Hub embedding model."""
    sent_hl_units: Union[int, List[int]] = 32
    """Number of units in the hidden layer."""
    sent_dropout: Union[float, List[float]] = 0.1
    """Dropout rate applied after the hidden layer."""
    l1_reg: float = 0.01
    """L1 regularization factor for the hidden layer kernel."""
    l2_reg: float = 0.01
    """L2 regularization factor for the hidden layer kernel."""
    epochs: int = 500
    """Maximum number of epochs for training."""
    callback_patience: int = 20
    """Number of epochs with no improvement to wait before early stopping."""
    learning_rate: Union[float, List[float]] = 5e-3
    """Initial learning rate for the optimizer."""
    validation_split: float = 0.2
    """Fraction of the training data to be used as validation data."""

def remove_duplicate_words(text: str) -> str:
    """
    Removes consecutive duplicate words from a string.

    :param text: The input string.
    :type text: str
    :return: The text with consecutive duplicates removed.
    :rtype: str
    """
    # Note: The original implementation removes ALL duplicate words, not just consecutive.
    # The docstring reflects the implementation.
    words = text.split()
    seen = set()
    result = []
    for word in words:
        if word not in seen:
            seen.add(word)
            result.append(word)
    return ' '.join(result)


def fetch_artifact_from_wandb(model_full_name: str) -> Tuple[str, str]:
    """
    Download a model artifact from W&B and return the paths to the model and config files.

    :param model_full_name: The W&B artifact full name (e.g., "adaj/intent-classifier-2025-2/confusion-clf:v1").
                           Must have format: "entity/project/artifact_name:version"
    :type model_full_name: str
    :return: A tuple containing the local file path to the Keras model file and the config file.
    :rtype: tuple[str, str]
    :raises ValueError: If format is invalid or files are not found in the artifact.
    """
    # Validate format
    parts = model_full_name.split("/")
    if len(parts) != 3 or ":" not in parts[2]:
        raise ValueError(
            f"Invalid model_full_name format: '{model_full_name}'. "
            f"Expected format: 'entity/project/artifact_name:version' (e.g., 'adaj/intent-classifier-2025-2/confusion-clf:v1')"
        )

    # Download artifact from W&B
    try:
        api = wandb.Api()
        artifact = api.artifact(model_full_name, type='model')
    except wandb.errors.CommError as e:
        raise ValueError(f"Could not fetch artifact '{model_full_name}' from W&B. Ensure the path is correct and you are logged in. Original error: {e}")

    # Create a target directory for the download
    models_dir = Path(os.path.dirname(__file__)) / "models"
    models_dir.mkdir(exist_ok=True)

    # Download artifact content. The path returned is the directory where files are.
    download_path = artifact.download(root=models_dir)

    model_file, config_file = None, None
    # Iterate over the files *in the artifact manifest* to find the correct ones.
    # This prevents accidentally loading unrelated files from the same directory.
    for f in artifact.files():
        if f.name.endswith((".keras", ".h5")):
            model_file = os.path.join(download_path, f.name)
        elif f.name.endswith("_config.yml"):
            config_file = os.path.join(download_path, f.name)

    if not model_file:
        raise ValueError(f"Model file (.keras or .h5) not found in W&B artifact '{model_full_name}'.")
    if not config_file:
        raise ValueError(f"Config file (_config.yml) not found in W&B artifact '{model_full_name}'.")

    return model_file, config_file


class IntentClassifier:
    """
    A class for training, evaluating, and predicting text intents using a Keras model.

    This classifier wraps the entire MLOps pipeline, including data loading,
    preprocessing, model building (using TensorFlow Hub embeddings), training,
    W&B logging, and prediction.

    :param config: A path to a YAML config file, a Config object, or None.
                   If None, config is inferred from `load_model`.
    :type config: str, Config, optional
    :param load_model: A path to a saved Keras model (`.keras` file) or a W&B artifact URL.
                       If provided, the model and its associated config are loaded.
    :type load_model: str, optional
    :param training_data: Path to a YAML file containing training examples.
                          Required for training or cross-validation.
    :type training_data: str, optional
    :raises ValueError: If `config` is not provided and cannot be inferred from `load_model`.
    :raises ValueError: If `load_model` path is provided but the file is not found.
    """

    def __init__(self, config: Optional[Union[str, Config]] = None,
                 load_model: Optional[str] = None,
                 training_data: Optional[str] = None,
                 wandb_project: Optional[str] = None):
        """
        Initializes the IntentClassifier.
        """
        self.model = None
        local_model_path = None

        # Set up W&B project early
        self.wandb_project = wandb_project or os.environ.get("WANDB_PROJECT")

        if load_model:
            # Check if load_model is a local file path or a W&B artifact URL
            if os.path.exists(load_model):
                local_model_path = load_model
            else:
                # If not a local path, assume it's a W&B artifact and fetch it.
                # The associated config path will be discovered and used automatically.
                local_model_path, config = fetch_artifact_from_wandb(load_model)

            self.model = tf.keras.models.load_model(local_model_path)
            print(f"Loaded Keras model from {local_model_path}.")

        # Load config. If fetched from W&B, `config` is already the correct path.
        self._load_config(config)

        # Load intents from the examples file if provided
        self._load_intents(training_data)

        # Validate model output size matches config codes (if model is loaded)
        if self.model:
            self._validate_model_config_compatibility()

        # Initialize stop_words and one-hot encoder
        self._load_stop_words(self.config.stop_words_file)
        self._setup_onehot_encoder()

        # Set up W&B run
        if self.wandb_project:
            print(f"Setting up W&B project: {self.wandb_project}")
            wandb.login(key=os.environ.get("WANDB_API_KEY"))
            self.wandb_run = wandb.init(project=self.wandb_project, config=self.config.__dict__)
            if self.training_data:
                artifact = wandb.Artifact(Path(self.training_data).name, type="dataset")
                artifact.add_file(self.training_data)
                self.wandb_run.log_artifact(artifact)
        else:
            self.wandb_run = None
            print("W&B project not set. No W&B run will be created.")

    def _load_config(self, config: Optional[Union[str, Config]]) -> None:
        """
        Loads the configuration from a file path or a Config object.

        :param config: A path to a YAML config file or a Config object.
        :type config: str, Config, optional
        :raises ValueError: If config is not provided or is of an invalid type.
        """
        if isinstance(config, str):
            with open(config, 'r') as f:
                self.config = Config(**yaml.safe_load(f))
            print(f"Loaded config from {config}.")
        elif isinstance(config, Config):
            self.config = config
        elif config is None:
            # If no config is provided at all, we can't proceed.
            raise ValueError(
                "A 'config' must be provided, either as a file path, a Config object, "
                "or as part of a W&B artifact."
            )
        else:
            raise TypeError(f"Unsupported config type: {type(config)}")

    def _load_intents(self, training_data: Optional[str]) -> np.ndarray:
        """
        Loads and preprocesses intents from a YAML examples file.

        If `training_data` is provided, it reads the file, extracts utterances
        and labels, shuffles them, and stores them in `self.input_text`
        and `self.labels`. It also populates `self.codes` and `self.config.codes`.

        If `training_data` is None, it loads `self.codes` from `self.config.codes`.

        :param training_data: Path to a YAML file with intent examples.
        :type training_data: str, optional
        :return: An array of unique intent codes (labels).
        :rtype: np.ndarray
        """
        self.training_data = training_data
        if training_data is not None:
            pprint(f"Loading intents from {training_data}...")
            with open(training_data, 'r') as f:
                self.intents_data = yaml.safe_load(f)
            # Preprocess intents
            input_text = []
            labels = []
            for i in self.intents_data:
                input_text += i['examples']
                labels += [i['intent']]*len(i['examples'])
            input_text = np.array(input_text)
            labels = np.array(labels)
            # Shuffle data
            indices = np.arange(len(labels))
            np.random.shuffle(indices)
            self.input_text = input_text[indices]
            self.input_text = tf.convert_to_tensor(self.input_text, dtype=tf.string)
            self.labels = labels[indices]
            self.codes = np.unique(self.labels)
            self.config.codes = self.codes.tolist()
        else: # Means that the example_file is not provided
            # Then the model will be used only to predict, no need to load training data
            self.codes = self.config.codes
        return self.codes

    def _load_stop_words(self, stop_words_file: Optional[str]) -> 'IntentClassifier':
        """
        Loads stopwords from a file.

        :param stop_words_file: Path to a text file containing stopwords, one per line.
        :type stop_words_file: str, optional
        :return: The IntentClassifier instance.
        :rtype: IntentClassifier
        """
        if stop_words_file is None:
            self.stop_words = []
            return self
        with open(stop_words_file, 'r', encoding='utf-8') as f:
            self.stop_words = f.read().split('\n')
        print(f"Loaded {len(self.stop_words)} stop words from {stop_words_file}.")
        return self

    def _validate_model_config_compatibility(self) -> None:
        """
        Validates that the loaded model's output size matches the number of categories
        in config.codes. This ensures the model and config are compatible.

        :raises ValueError: If the model's output size doesn't match the number of codes
                            in the config, indicating an invalid model-config mismatch.
        """
        if self.model is None:
            return
        # Get the model's output layer size (number of classes the model was trained with)
        model_output_size = self.model.output_shape[-1]
        # Get the number of codes from config
        config_codes_count = len(self.config.codes) if self.config.codes else 0
        # Also check onehot_encoder categories if it exists (though it shouldn't at this point)
        # This is a double-check to ensure consistency
        if model_output_size != config_codes_count:
            error_msg = (
                f"Model-config mismatch detected: The loaded model was trained with "
                f"{model_output_size} categories, but the config file specifies "
                f"{config_codes_count} categories (codes: {self.config.codes}).\n\n"
                f"This indicates an invalid model configuration. The model's output layer "
                f"and the one-hot encoder categories must match.\n\n"
            )
            raise ValueError(error_msg)

    def _setup_onehot_encoder(self) -> OneHotEncoder:
        """
        Initializes and fits the OneHotEncoder based on the loaded intent codes.

        :return: The fitted OneHotEncoder instance.
        :rtype: OneHotEncoder
        """
        assert self.codes is not None, "cfrom google.colab import userdatades must be set before setting up the encoder."
        self.onehot_encoder = OneHotEncoder(categories=[self.codes],)\
                                  .fit(np.array(self.codes).reshape(-1, 1))
        return self.onehot_encoder

    def _get_callbacks(self) -> list:
        """
        Constructs a list of Keras callbacks based on the configuration.

        Includes EarlyStopping and WandbMetricsLogger if configured.
        Also includes an ExponentialDecay learning rate scheduler.

        :return: A list of Keras Callback instances.
        :rtype: list
        """
        callbacks = []
        if self.config.callback_patience > 0:
            callbacks.append(
                tf.keras.callbacks.EarlyStopping(monitor='val_f1_score',
                    patience=self.config.callback_patience,
                    restore_best_weights=True)
            )
        if self.wandb_project:
            callbacks.append(WandbMetricsLogger())

        # Configure ExponentialDecay
        if self.config.learning_rate is not None and not isinstance(self.config.learning_rate, str):
            lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
                initial_learning_rate=self.config.learning_rate,
                decay_steps=1000,
                decay_rate=0.96,
                staircase=False
            )

            # Modified learning rate scheduler to properly handle epoch parameter
            def lr_scheduler(epoch, lr):
                """Internal LR scheduler function."""
                return lr_schedule(epoch).numpy().astype(float)

            lr_scheduler_callback = tf.keras.callbacks.LearningRateScheduler(lr_scheduler)
            callbacks.append(lr_scheduler_callback)
        return callbacks

    def finish_wandb(self):
        """
        Finishes the current Weights & Biases run, if one is active.
        """
        if self.wandb_project and self.wandb_run:
            self.wandb_run.finish()

    def preprocess_text(self, text: tf.Tensor) -> tf.Tensor:
        """
        Applies preprocessing steps to a single raw text tensor.

        Steps include:
        1. Lowercasing.
        2. Stopword removal (if configured).
        3. Padding with "<>" tokens if text is shorter than `min_words`.

        :param text: A 0-D string tensor (scalar) containing the raw text.
        :type text: tf.Tensor
        :return: A 0-D string tensor (scalar) containing the preprocessed text.
        :rtype: tf.Tensor
        """
        text = tf.strings.lower(text)
        if self.stop_words:
            words = tf.strings.split(text)
            # Create a mask for words that are NOT in stopwords
            words = tf.boolean_mask(words, tf.reduce_all(tf.not_equal(words[:, None], tf.constant(self.stop_words)), axis=1))
            text = tf.strings.reduce_join(words, separator=' ')

        if self.config.min_words:
            # Remove punctuation before counting words
            words = tf.strings.split(text)
            words = tf.boolean_mask(words, tf.reduce_all(tf.not_equal(words[:, None], tf.constant(["?", ".", ",", "!"])), axis=1))
            num_words = tf.shape(words)[0]

            # Check if num_words is less than or equal to min_words
            if tf.less_equal(num_words, self.config.min_words):
                # Create padding
                padding = tf.strings.join(["<>"] * (self.config.min_words + 1), separator=' ')
                text = padding

        # Iterate on text and replace punctuation with "PUNCTUATION" (it helps some sentence encoders that do not parse punctuation, like Universal Sentence Encoder)
        for p, t in {"?": "QUESTION_MARK", ".": "PERIOD", ",": "COMMA", "!": "EXCLAMATION_MARK"}.items():
            espaped_p = re.escape(p)
            text = tf.strings.regex_replace(text, espaped_p, f" {t} ")
        text = tf.strings.regex_replace(text, r"\s+", " ")
        text = tf.strings.strip(text)

        # Ensure output is always a 0-D tensor (scalar)
        return tf.strings.as_string(text)

    def make_model(self, config: Config) -> tf.keras.Model:
        """
        Builds and returns a new Keras model based on the provided configuration.

        The architecture consists of:
        1. An input layer for string tensors.
        2. A `HubLayer` for text embedding.
        3. A Dense hidden layer with BatchNormalization, ReLU activation, and Dropout.
        4. A final Dense output layer with softmax activation for classification.

        :param config: The configuration object specifying model hyperparameters.
        :type config: Config
        :return: A compiled Keras model.
        :rtype: tf.keras.Model
        """
        # Set the random seed for reproducibility
        seed = 42
        tf.random.set_seed(seed)  # Assuming you have a random_seed in your config
        # Extract config values
        sent_hl_units, sent_dropout = config.sent_hl_units, config.sent_dropout
        l1_reg, l2_reg = config.l1_reg, config.l2_reg
        output_size = len(self.codes)
        # Build model
        initializer = tf.keras.initializers.GlorotUniform(seed=seed)  # Set seed in initializer
        text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name="inputs")
        # Sentence encoder
        encoder = HubLayer(config.embedding_model, trainable=False, name="sent_encoder")(text_input)
        # Hidden layer
        sent_hl = tf.keras.layers.Dense(sent_hl_units,
                                        kernel_initializer=initializer,
                                        kernel_regularizer=regularizers.l1_l2(l1=l1_reg, l2=l2_reg),
                                        activation=None,  # No activation here yet
                                        name='sent_hl')(encoder)
        sent_hl_norm = tf.keras.layers.BatchNormalization()(sent_hl)  # Add batch normalization
        sent_hl_activation = tf.keras.layers.Activation('relu')(sent_hl_norm)  # Activation after batch normalization
        sent_hl_dropout = tf.keras.layers.Dropout(sent_dropout, seed=seed)(sent_hl_activation)  # Set seed in dropout
        # Output layer
        sent_output = tf.keras.layers.Dense(output_size,
                                            kernel_initializer=initializer,
                                            activation='softmax',
                                            name="sent_output")(sent_hl_dropout)
        model = tf.keras.Model(inputs=text_input, outputs=sent_output)
        return model

    def train(self, save_model: Optional[str] = None, tf_verbosity: int = 1) -> tf.keras.Model:
        """
        Trains the model on the loaded examples.

        This method splits the data, preprocesses it, builds a new model,
        compiles it, and runs the training loop. If `save_model` is provided,
        the trained model and its config are saved.

        :param save_model: Path to save the trained model (`.keras` format).
                           Config will be saved alongside (e.g., `model_config.yml`).
        :type save_model: str, optional
        :param tf_verbosity: Verbosity level for Keras `fit` method (0, 1, or 2).
        :type tf_verbosity: int, optional
        :return: The trained Keras model.
        :rtype: tf.keras.Model
        :raises AssertionError: If `training_data` was not provided during initialization.
        """
        pprint(self.config.__dict__)
        # Update task config parameter
        self.config.task = "train"
        assert self.training_data is not None, "training_data must be provided when the IntentClassifier was created."

        # Extract one-hot encoded labels
        labels_ohe = self.onehot_encoder\
                            .transform(self.labels.reshape(-1, 1))\
                            .toarray()
        # Split
        X_train_text, X_val_text, y_train, y_val = train_test_split(
            self.input_text.numpy(), labels_ohe, # Convert to NumPy array for splitting
            test_size=self.config.validation_split,
            stratify=labels_ohe,      # Ensure class distribution is preserved
            random_state=42           # For reproducibility
        )
        # Now apply preprocessing using preprocess_text *after* splitting:
        X_train = tf.map_fn(self.preprocess_text, tf.constant(X_train_text), dtype=tf.string)
        X_val = tf.map_fn(self.preprocess_text, tf.constant(X_val_text), dtype=tf.string)

        # Extract config values
        epochs = self.config.epochs
        # New model from scratch
        self.model = self.make_model(self.config)
        self.model.compile(
            loss='categorical_crossentropy',
            optimizer=tf.keras.optimizers.Adam(), # LR is handled by callback
            metrics=[tf.keras.metrics.F1Score(average='macro')])
        # Train the model
        self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            # batch_size=16,
            shuffle=True,
            epochs=epochs,
            verbose=tf_verbosity,
            callbacks=self._get_callbacks()
        )
        # Save model
        if save_model is not None:
            self.save_model(path=save_model)
        return self.model

    def save_model(self, path: str):
        """
        Saves the current model and its configuration file.

        The model is saved in Keras format (`.keras`).
        The config is saved as a YAML file with `_config.yml` suffix.
        If W&B is configured, the model is also logged as an artifact.

        :param path: The base path to save the model (e.g., "models/my_model.keras").
        :type path: str
        """
        Path(os.path.dirname(path)).mkdir(parents=True, exist_ok=True)
        # Save model in SavedModel format
        if path.endswith('/'):
            # Remove trailing slash if present
            path = path.rstrip('/')
        # Save the model
        self.model.save(path)
        # Save config into a yaml file inside the model directory
        config_path = path.replace(".keras", "_config.yml") #os.path.join(os.path.dirname(path), f"{self.config.dataset_name}_config.yml")
        with open(config_path, 'w') as f:
            f.write(yaml.dump(self.config.__dict__))
        print(f"Model saved to {path}.")
        if self.wandb_project:
            # Crie e envie o artifact
            artifact = wandb.Artifact(
                name=f"{self.config.dataset_name}-clf",
                type="model",
                description="Modelo Keras para classificação de intenção"
            )
            artifact.add_file(path)
            artifact.add_file(config_path) # Also add the config file
            self.wandb_run.log_artifact(artifact)
            self.finish_wandb() # Finish the run after saving

    def predict(self, input_text: Union[str, List[str]],
                true_labels: Optional[List[str]] = None,
                log_to_wandb: bool = False) -> Union[Tuple[str, Dict[str, float]], List[Tuple[str, Dict[str, float]]]]:
        """
        Predicts the intent for a given text or list of texts.

        :param input_text: A single text string or a list of text strings to classify.
        :type input_text: str or list[str]
        :param true_labels: A list of true labels, corresponding to `input_text`. Used for logging to W&B.
        :type true_labels: list[str], optional
        :param log_to_wandb: If True, logs the inputs, predictions, and true labels (if provided) to W&B.
        :type log_to_wandb: bool, optional
        :return: If `input_text` is a string: a tuple `(top_intent, all_probabilities)`.
                 If `input_text` is a list: a list of tuples `[(top_intent, all_probabilities), ...]`.
                 `all_probabilities` is a dict mapping intent codes to their predicted probabilities.
        :rtype: tuple(str, dict(str, float)) or list[tuple(str, dict(str, float))]
        """
        self.config.task = "predict"  # Set the task to "predict"
        original_input_is_string = isinstance(input_text, str)
        if original_input_is_string:
            input_text_list = [input_text]  # Convert single string to a list for processing
        else:
            input_text_list = input_text

        # Preprocess each string in the list and stack them
        preprocessed_texts = tf.map_fn(self.preprocess_text, tf.constant(input_text_list), dtype=tf.string)

        # Predict probabilities for all strings at once
        all_probs = self.model.predict(preprocessed_texts)
        results = []
        predicted_labels_for_log = []
        for i in range(all_probs.shape[0]):
            current_probs = all_probs[i] # Probabilities for the i-th input text
            # Determine the intent name with the highest probability
            highest_prob_idx = np.argmax(current_probs)
            highest_prob_intent_name = self.codes[highest_prob_idx]
            predicted_labels_for_log.append(highest_prob_intent_name)
            # Create a dictionary of probabilities for each intent name
            probs_dict = {code: float(current_probs[j]) for j, code in enumerate(self.codes)}
            results.append((highest_prob_intent_name, probs_dict))

        # Log to Wandb if requested
        if log_to_wandb and self.wandb_project:
            # Get the current run ID if it exists, otherwise start a new run
            run_id = wandb.run.id if wandb.run else wandb.util.generate_id()
            # Initialize wandb with the run ID
            with wandb.init(project=self.wandb_project, id=run_id, resume="allow"):
                wandb.log({
                    "inputs": input_text_list, # Log the list of original input texts
                    "true_labels": true_labels,
                    "predictions": predicted_labels_for_log # Use the extracted list of highest prob intents
                })

        # Return a single tuple if the original input was a string, otherwise a list of tuples
        if original_input_is_string:
            return results[0]
        return results


/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Configuração do IntentClassifier

* dataset_name: str
* codes : List[str] = None
* architecture: str = "v0.1.5"
* stop_words_file: Optional[str] = None
* wandb_project: Optional[str] = None
* min_words: int = 1
* embedding_model: Union[str, List[str]] = 'https://www.kaggle.com/models/google/universal-sentence-encoder/tensorFlow2/multilingual/2'
* sent_hl_units: Union[int, List[int]] = 32
* sent_dropout: Union[float, List[float]] = 0.1
* l1_reg: float = 0.01
* l2_reg: float = 0.01
* epochs: int = 500
* callback_patience: int = 20
* learning_rate: Union[float, List[float]] = 5e-3
* validation_split: float = 0.2

### a. Treinando intent classifier para detectar **Confusão/Certeza/Neutro**.

Carregue o arquivo "confusion_examples.yml" e rode a seguinte célula para treinar o modelo e salvá-lo na aba Arquivos.

In [ ]:
# from intent_classifier import Config, IntentClassifier

In [ ]:
from google.colab import userdata
import wandb

wandb.login(userdata.get('WANDB_KEY'))

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adaj to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
! wandb agent adaj/mlops-2026-2-2/6bz35ixp

wandb: Starting wandb agent 🕵️
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
2026-09-18 00:50:10,086 - wandb.wandb_agent - INFO - Running runs: []
2026-09-18 00:50:10,657 - wandb.wandb_agent - INFO - Agent received command: run
2026-09-18 00:50:10,657 - wandb.wandb_agent - INFO - Agent starting run with config:
	sent_dropout: 0.32106430305667305
	sent_hl_units: 95
2026-09-18 00:50:10,658 - wandb.wandb_agent - INFO - About to run command: /usr/bin/env python train.py --sent_dropout=0.32106430305667305 --sent_hl_units=95
python3: can't open file '/content/train.py': [Errno 2] No such file or directory
2026-09-18 00:50:15,664 - wandb.wandb_agent - INFO - Running runs: ['7o2bp08o']
2026-09-18 00:50:15,665 - wandb.wandb_agent - INFO - Cleaning up finished run: 7o2bp08o
2026-09-18 00:50:16,367 - wandb.wandb_agent - INFO - Agent received command: run
2026-09-18 00:50:16,367 - wandb.wandb_agent - INFO - Agent starting run with config:
	sent_dropout: 0.60

In [ ]:
# Setup training parameters
training_config = Config(
    dataset_name="confusion",
    sent_hl_units=16,
    sent_dropout=0.6,
    learning_rate=0.0001,
    epochs=500,
    callback_patience=100,
    validation_split=0.20
)

# Initialize the classifier
clf = IntentClassifier(config=training_config,
                       training_data='confusion_examples.yml',
                       wandb_project="mlops-2026-2-2")

# Start training
clf.train(save_model="model.h5")

'Loading intents from confusion_examples.yml...'
Setting up W&B project: mlops-2026-2-2


wandb: WARNING Artifact "confusion_examples.yml" already exists with the same content. No new version will be created.


{'architecture': 'v0.1.5',
 'callback_patience': 100,
 'codes': ['certainty', 'confusion', 'neutral_statement'],
 'dataset_name': 'confusion',
 'embedding_model': 'https://www.kaggle.com/models/google/universal-sentence-encoder/tensorFlow2/multilingual/2',
 'epochs': 500,
 'l1_reg': 0.01,
 'l2_reg': 0.01,
 'learning_rate': 0.0001,
 'min_words': 1,
 'sent_dropout': 0.6,
 'sent_hl_units': 16,
 'stop_words_file': None,
 'task': 'undefined',
 'validation_split': 0.2}
Epoch 1/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 11s 566ms/step - f1_score: 0.2868 - loss: 6.4105 - val_f1_score: 0.2579 - val_loss: 5.7526 - learning_rate: 1.0000e-04
Epoch 2/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - f1_score: 0.2647 - loss: 6.1891 - val_f1_score: 0.2579 - val_loss: 5.7371 - learning_rate: 9.9996e-05
Epoch 3/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - f1_score: 0.1990 - loss: 6.3447 - val_f1_score: 0.2913 - val_loss: 5.7219 - learning_rate: 9.9992e-05
Epoch 4/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 153ms/step - f1_score: 0.245

Model saved to model.h5.


epoch/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇██
epoch/f1_score,▁▃▄▄▅▄▅▆▅▅▆▆▆▇▆▇▇▇▇▇▇▇▇▇█▇▇▇█▇█▇█▇█▇██▇█
epoch/learning_rate,█████▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁
epoch/loss,█▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁
epoch/val_f1_score,▁▁▂▂▃▅▅▅▅▅▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████
epoch/val_loss,████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁
epoch/epoch,315
epoch/f1_score,0.83917
epoch/learning_rate,0.0001
epoch/loss,1.26298
epoch/val_f1_score,0.79649


<Functional name=functional_6, built=True>

In [ ]:
# .train
# .predict

In [ ]:
clf.predict(["teste"])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step


[(np.str_('neutral_statement'),
  {np.str_('certainty'): 0.11065290123224258,
   np.str_('confusion'): 0.21632954478263855,
   np.str_('neutral_statement'): 0.6730175614356995})]

### a. Treinando intent classifier para detectar perguntas ao chatbot Clair.

Carregue o arquivo "clair_intents_example.yml" e rode a seguinte célula para treinar o modelo e salvá-lo na aba Arquivos (para isso, altere a variável `save_model` abaixo).

In [ ]:
examples_file = 'clair_intents_examples.yml'
save_model = None

# Setup training parameters
training_config = Config(
    wandb_project="intent-classifier-2025-2",
    dataset_name="clair_intents",
    sent_hl_units=64,
    sent_dropout=0.5,
    learning_rate=0.0005,
    epochs=200,
    callback_patience=20,
    validation_split=0.1
)


# Initialize the classifier
clf = IntentClassifier(config=training_config,
                       examples_file=examples_file)

# Start training
clf.train(save_model=save_model)

In [ ]:
clf.predict(['hi clair'], get_certainty='all')

In [ ]:
exemplos_novos = [
    "oi amigos",  # normal_response
    "tchau tchau", # normal_response
    "vamos para questão 3", # normal_response
    "clair vc pode me ajudar?", # ask_support
    "clair ta ai?", # ping
    "clair me ajuda com a questão 3?", # normal_response
    "alguem me ajuda na questão 3" # normal_response
]

clf.predict(exemplos_novos)

In [ ]:
clf2.predict(['eu estou certo de que estou confuso'], get_certainty='all')

In [ ]:
exemplos_novos2 = [
    "oi amigos",  # neutral_statement
    "tchau tchau", # neutral_statement
    "vamos para questão 3", # neutral_statement
    "clair vc pode me ajudar?", # confusion
    "clair ta ai?", # neutral_statement
    "clair me ajuda com a questão 3?", # confusion
    "alguem me ajuda na questão 3" # confusion
    # Mais outras
    "me sinto confiante que a resposta é 10", # certainty
    "acho que a resposta é essas mesmo", # certainty
    "não faço ideia do que fazer agora", # confusion
    "alguem sabe oq fazer?", # confusion
    "não tou entendendo nada" # confusion
]

clf2.predict(exemplos_novos2)

## Tarefa

Instruções:

1. Explore a implementação e uso da classe IntentClassifier, especialmente as funções train e predict.

2. Insira os arquivos "clair_intents_examples.yml" e "confusion_examples.yml" nos Arquivos do notebook.

3. Treine modelos para cada um desses arquivos utilizando IntentClassifier.train() e avalie em novos exemplos utilizando IntentClassifier.predict()

4. Monitore o treinamento dos modelos no W&B e reflita sobre o resultado.

5. Explique quais funcionalidades do W&B podem ser de grande utilidade para desenvolver modelos otimizados.
